In [56]:
import pandas
from pyspark.sql.functions import * 
from pyspark.sql.window import Window
from pyspark.sql import SparkSession

In [57]:
spark = SparkSession.builder \
    .appName("Read_From_MinIO") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.1") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "12345678") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

In [67]:
df_crm_cust = spark.read \
    .option("header", True) \
    .parquet("s3a://silver/crm/cust/dt=20260430/")

df_crm_prd = spark.read \
    .option("header", True) \
    .parquet("s3a://silver/crm/prd/dt=20260430/")


df_crm_sales = spark.read \
    .option("header", True) \
    .parquet("s3a://silver/crm/sales/dt=20260430/")

df_erp_cat = spark.read \
    .option("header", True) \
    .parquet("s3a://silver/erp/cat/dt=20260430/")

df_erp_cust = spark.read \
    .option("header", True) \
    .parquet("s3a://silver/erp/cust/dt=20260430/")

df_erp_loc = spark.read \
    .option("header", True) \
    .parquet("s3a://silver/erp/loc/dt=20260430/")



silver_erp_cust
silver_crm_cust
silver_erp_loc 
--> dim_customers

In [66]:
df_erp_cust.show(3)
print ('crm cust')
df_crm_cust.show(3)
print ('erp_loc')
df_erp_loc.show(3)

+----------+----------+----+----------+
|       CID|     BDATE| GEN| load_date|
+----------+----------+----+----------+
|AW00011000|1971-10-06|MALE|2026-05-01|
|AW00011001|1976-05-10|MALE|2026-05-01|
|AW00011002|1971-02-09|MALE|2026-05-01|
+----------+----------+----+----------+
only showing top 3 rows

crm cust
+------+----------+-------------+------------+------------------+--------+---------------+----------+
|cst_id|   cst_key|cst_firstname|cst_lastname|cst_marital_status|cst_gndr|cst_create_date| load_date|
+------+----------+-------------+------------+------------------+--------+---------------+----------+
| 11000|AW00011000|          Jon|       Yang |           Married|    Male|     2025-10-06|2026-05-01|
| 11001|AW00011001|       Eugene|     Huang  |            Single|    Male|     2025-10-06|2026-05-01|
| 11002|AW00011002|        Ruben|      Torres|           Married|    Male|     2025-10-06|2026-05-01|
+------+----------+-------------+------------+------------------+--------+

In [79]:
df_erp_cust.show(1)
df_crm_cust.show(1)
df_erp_loc.show(1)

+----------+----------+----+----------+
|       CID|     BDATE| GEN| load_date|
+----------+----------+----+----------+
|AW00011000|1971-10-06|MALE|2026-05-01|
+----------+----------+----+----------+
only showing top 1 row

+------+----------+-------------+------------+------------------+--------+---------------+----------+
|cst_id|   cst_key|cst_firstname|cst_lastname|cst_marital_status|cst_gndr|cst_create_date| load_date|
+------+----------+-------------+------------+------------------+--------+---------------+----------+
| 11000|AW00011000|          Jon|       Yang |           Married|    Male|     2025-10-06|2026-05-01|
+------+----------+-------------+------------+------------------+--------+---------------+----------+
only showing top 1 row

+----------+---------+----------+
|       CID|    CNTRY| load_date|
+----------+---------+----------+
|AW00011000|AUSTRALIA|2026-05-01|
+----------+---------+----------+
only showing top 1 row



In [118]:
df_crm_cust.select(col('cst_gndr')).distinct().show()

+--------+
|cst_gndr|
+--------+
|  Female|
|     N/a|
|    Male|
+--------+



In [124]:
df_dim_customers = df_crm_cust.alias("crm_c").join(df_erp_cust.alias("erp_c"), col("crm_c.cst_key") == col("erp_c.CID") , "left")\
            .join(df_erp_loc.alias("erp_l"), col("crm_c.cst_key") == col("erp_l.CID"), "left")\
            .select (
                col("crm_c.cst_id"),\
                col("crm_c.cst_key"),\
                col("crm_c.cst_firstname"),\
                col("crm_c.cst_lastname"),\
                col("erp_l.CNTRY"),\
                col("crm_c.cst_marital_status"),\
                # when (col("crm_c.cst_gndr") != "N/a", col("crm_c.cst_gndr")).otherwise(col("erp.GEN"))
                when(
                    col("crm_c.cst_gndr") != "N/a",
                    col("crm_c.cst_gndr")
                ).otherwise(col("erp_c.GEN")).alias("cst_gndr"),\
                col("erp_c.bdate"),\
                col("crm_c.cst_create_date"),\
            ).show()


# select(
#     col("a.id"),
#     col("a.name"),
#     col("b.address")
# )
# CREATE VIEW gold.dim_customers AS
# SELECT
#     ROW_NUMBER() OVER (ORDER BY cst_id) AS customer_key, -- Surrogate key
#     ci.cst_id                          AS customer_id,
#     ci.cst_key                         AS customer_number,
#     ci.cst_firstname                   AS first_name,
#     ci.cst_lastname                    AS last_name,
#     la.cntry                           AS country,
#     ci.cst_marital_status              AS marital_status,
#     CASE 
#         WHEN ci.cst_gndr != 'n/a' THEN ci.cst_gndr -- CRM is the primary source for gender
#         ELSE COALESCE(ca.gen, 'n/a')  			   -- Fallback to ERP data
#     END                                AS gender,
#     ca.bdate                           AS birthdate,
#     ci.cst_create_date                 AS create_date
# FROM silver.crm_cust_info ci
# LEFT JOIN silver.erp_cust_az12 ca
#     ON ci.cst_key = ca.cid
# LEFT JOIN silver.erp_loc_a101 la
#     ON ci.cst_key = la.cid;
# GO

+------+----------+-------------+------------+---------+------------------+--------+----------+---------------+
|cst_id|   cst_key|cst_firstname|cst_lastname|    CNTRY|cst_marital_status|cst_gndr|     bdate|cst_create_date|
+------+----------+-------------+------------+---------+------------------+--------+----------+---------------+
| 11000|AW00011000|          Jon|       Yang |AUSTRALIA|           Married|    Male|1971-10-06|     2025-10-06|
| 11001|AW00011001|       Eugene|     Huang  |AUSTRALIA|            Single|    Male|1976-05-10|     2025-10-06|
| 11002|AW00011002|        Ruben|      Torres|AUSTRALIA|           Married|    Male|1971-02-09|     2025-10-06|
| 11003|AW00011003|      Christy|         Zhu|AUSTRALIA|            Single|  Female|1973-08-14|     2025-10-06|
| 11004|AW00011004|    Elizabeth|     Johnson|AUSTRALIA|            Single|  Female|1979-08-05|     2025-10-06|
| 11005|AW00011005|        Julio|        Ruiz|AUSTRALIA|            Single|    Male|1976-08-01|     2025

silver_crm_prd
silver erp_cat
--> dim_product


In [ ]:
CREATE VIEW gold.dim_products AS
SELECT
    ROW_NUMBER() OVER (ORDER BY pn.prd_start_dt, pn.prd_key) AS product_key, -- Surrogate key
    pn.prd_id       AS product_id,
    pn.prd_key      AS product_number,
    pn.prd_nm       AS product_name,
    pn.cat_id       AS category_id,
    pc.cat          AS category,
    pc.subcat       AS subcategory,
    pc.maintenance  AS maintenance,
    pn.prd_cost     AS cost,
    pn.prd_line     AS product_line,
    pn.prd_start_dt AS start_date
FROM silver.crm_prd_info pn
LEFT JOIN silver.erp_px_cat_g1v2 pc
    ON pn.cat_id = pc.id
WHERE pn.prd_end_dt IS NULL; -- Filter out all historical data
GO

crm_sales
dim_products
dim_customers 
--> fact_sales